In [ ]:
import pandas as pd
import glob
import numpy as np
from tqdm import tqdm
import os

In [ ]:
num_layers = 58
num_experts = 256
top_k = 8

In [ ]:
data = np.load('data/lmsys/activation_counts_top1.npz')
stats = list(data.values())
len(stats)

In [ ]:
def estimate_benefit(exp_count):
  mx = np.max(exp_count, axis=1)
  avg = np.mean(exp_count, axis=1)
  total_act = np.sum(exp_count, axis=1)[0].item()
  return (mx - avg) / total_act

benefits = list(map(estimate_benefit, stats))
benefits = np.stack(benefits)
print(benefits.mean())
print(benefits.mean(0))
benefits.mean(1).std()

In [ ]:
total_load = np.zeros_like(stats[0], dtype=int)
def estimate_kvload(exp_count):
  total_tokens = np.sum(exp_count, axis=1)[0].item() // top_k
  affine_exp = np.argmax(exp_count, axis=1)
  np.add.at(total_load, (np.arange(num_layers), affine_exp), total_tokens)

for exp_count in stats:
  estimate_kvload(exp_count)

total_load

In [ ]:
# heatmap 
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
plt.figure(figsize=(12, 8))
sns.heatmap(total_load+1, annot=False, fmt='d', cmap='viridis', cbar_kws={'label': 'Total Load'}, norm=LogNorm())
plt.title('Total Load per Layer and Expert')
plt.xlabel('Expert ID')
plt.ylabel('Layer ID')
plt.tight_layout()
plt.show()

In [ ]:
# CDF for total_load

plt.figure(figsize=(10, 6))
plt.hist(total_load.flatten(), bins=1000, cumulative=True, density=True, color='blue', alpha=0.7)
plt.title('CDF of Total Load per Layer and Expert')
plt.xlabel('Total Load')
plt.ylabel('Cumulative Density')
plt.grid(True)
# plt.xscale('log')
# plt.yscale('log')
plt.tight_layout()
plt.show()